# Credit Card Approval Prediction
### Using Machine Learning — Logistic Regression | Decision Tree | Random Forest | XGBoost

**Dataset:** Kaggle — Credit Card Approval Prediction (`rikdifos/credit-card-approval-prediction`)

**Objective:** Predict whether a credit card application will be approved or rejected based on applicant demographic and financial information.

---
**Workflow:**
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Target Variable Engineering
4. Feature Engineering
5. Preprocessing (Encoding + Scaling)
6. Model Training & Evaluation
7. Model Comparison & Best Model Selection
8. Save Artifacts

## 1. Setup & Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

RANDOM_STATE = 42
print('Libraries loaded successfully.')

## 2. Data Loading & Overview

In [ ]:
# Update path if running notebook from a different directory
DATA_DIR = '../data'

app_df    = pd.read_csv(os.path.join(DATA_DIR, 'application_record.csv'))
credit_df = pd.read_csv(os.path.join(DATA_DIR, 'credit_record.csv'))

print(f'Application records : {app_df.shape[0]:,} rows × {app_df.shape[1]} columns')
print(f'Credit records      : {credit_df.shape[0]:,} rows × {credit_df.shape[1]} columns')

In [ ]:
print('=== Application Record — First 5 rows ===')
app_df.head()

In [ ]:
print('=== Application Record — Data Types & Non-null Counts ===')
app_df.info()

In [ ]:
print('=== Application Record — Descriptive Statistics ===')
app_df.describe().T

In [ ]:
print('=== Credit Record — STATUS value counts ===')
status_counts = credit_df['STATUS'].value_counts().sort_index()
print(status_counts)
print('\nSTATUS Legend:')
legend = {'0':'1-29 days past due','1':'30-59 days past due','2':'60-89 days past due',
          '3':'90-119 days past due','4':'120-149 days past due','5':'>150 days / bad debt',
          'C':'Paid off that month','X':'No loan that month'}
for k, v in legend.items():
    print(f'  {k}: {v}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Missing values heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

miss_app = app_df.isnull().sum()
miss_app = miss_app[miss_app > 0]
if len(miss_app):
    miss_app.sort_values().plot(kind='barh', ax=axes[0], color='#e74c3c')
    axes[0].set_title('Missing Values — Application Record', fontweight='bold')
    axes[0].set_xlabel('Count')
else:
    axes[0].text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('Missing Values — Application Record', fontweight='bold')

miss_credit = credit_df.isnull().sum()
miss_credit = miss_credit[miss_credit > 0]
if len(miss_credit):
    miss_credit.sort_values().plot(kind='barh', ax=axes[1], color='#e74c3c')
    axes[1].set_title('Missing Values — Credit Record', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
    axes[1].set_title('Missing Values — Credit Record', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Demographic distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Gender
app_df['CODE_GENDER'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#1a3a6b','#e74c3c'], edgecolor='white')
axes[0,0].set_title('Gender Distribution', fontweight='bold')
axes[0,0].set_xticklabels(['Male','Female'], rotation=0)

# Age distribution
ages = (-app_df['DAYS_BIRTH'] / 365).astype(int)
axes[0,1].hist(ages, bins=20, color='#2d5aa8', edgecolor='white')
axes[0,1].set_title('Age Distribution', fontweight='bold')
axes[0,1].set_xlabel('Age (years)')

# Income distribution
axes[0,2].hist(app_df['AMT_INCOME_TOTAL'], bins=30, color='#27ae60', edgecolor='white')
axes[0,2].set_title('Annual Income Distribution', fontweight='bold')
axes[0,2].set_xlabel('Income (₹)')
axes[0,2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e5:.0f}L'))

# Income Type
app_df['NAME_INCOME_TYPE'].value_counts().plot(kind='barh', ax=axes[1,0], color='#8e44ad', edgecolor='white')
axes[1,0].set_title('Income Type', fontweight='bold')

# Education Type
app_df['NAME_EDUCATION_TYPE'].value_counts().plot(kind='barh', ax=axes[1,1], color='#c8a84b', edgecolor='white')
axes[1,1].set_title('Education Type', fontweight='bold')

# Housing Type
app_df['NAME_HOUSING_TYPE'].value_counts().plot(kind='barh', ax=axes[1,2], color='#e74c3c', edgecolor='white')
axes[1,2].set_title('Housing Type', fontweight='bold')

plt.suptitle('Applicant Demographic Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric columns
numeric_cols = ['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
                'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS']

plt.figure(figsize=(10, 7))
corr = app_df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 4. Target Variable Engineering

In [ ]:
# STATUS '2','3','4','5' = bad (60+ days past due) → TARGET=0 (Rejected)
# All others → TARGET=1 (Approved)

BAD_STATUSES = {'2', '3', '4', '5'}

def applicant_label(statuses):
    return 0 if any(str(s) in BAD_STATUSES for s in statuses) else 1

target_df = (
    credit_df.groupby('ID')['STATUS']
    .apply(applicant_label)
    .reset_index()
    .rename(columns={'STATUS': 'TARGET'})
)

print('Target Distribution:')
print(target_df['TARGET'].value_counts())
print(f'\nApproval rate : {target_df["TARGET"].mean()*100:.1f}%')
print(f'Rejection rate: {(1 - target_df["TARGET"].mean())*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
labels = ['Approved (1)', 'Rejected (0)']
sizes  = target_df['TARGET'].value_counts().sort_index(ascending=False).values
colors = ['#27ae60', '#e74c3c']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, pctdistance=0.8,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Approval vs Rejection Distribution', fontweight='bold')

# STATUS distribution
status_counts = credit_df['STATUS'].value_counts().sort_index()
bar_colors = ['#27ae60' if s in ['C','X','0','1'] else '#e74c3c'
              for s in status_counts.index]
axes[1].bar(status_counts.index.astype(str), status_counts.values,
            color=bar_colors, edgecolor='white')
axes[1].set_title('Monthly Payment STATUS Distribution', fontweight='bold')
axes[1].set_xlabel('STATUS Code')
axes[1].set_ylabel('Count')

from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#27ae60', label='Good (C,X,0,1)'),
              Patch(facecolor='#e74c3c', label='Bad (2,3,4,5)')]
axes[1].legend(handles=legend_els)

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Merge application data with target
df = app_df.merge(target_df, on='ID', how='inner')
df.drop(columns=['ID'], inplace=True)
print(f'Merged dataset: {df.shape[0]:,} records')

# Feature engineering
df['AGE_YEARS']       = (-df['DAYS_BIRTH'] / 365).astype(int)
df['IS_UNEMPLOYED']   = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['YEARS_EMPLOYED']  = df['DAYS_EMPLOYED'].apply(lambda x: 0.0 if x == 365243 else round(-x / 365, 2))
df['INCOME_PER_MEMBER'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].clip(lower=1)

df['CODE_GENDER']     = (df['CODE_GENDER']    == 'F').astype(int)
df['FLAG_OWN_CAR']    = (df['FLAG_OWN_CAR']   == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)

df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)
print('\nNew features created: AGE_YEARS, IS_UNEMPLOYED, YEARS_EMPLOYED, INCOME_PER_MEMBER')
df[['AGE_YEARS','IS_UNEMPLOYED','YEARS_EMPLOYED','INCOME_PER_MEMBER','TARGET']].head()

In [ ]:
# Feature distributions by approval status
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title, color_map in zip(
    axes,
    ['AGE_YEARS', 'AMT_INCOME_TOTAL', 'YEARS_EMPLOYED'],
    ['Age by Approval Status', 'Income by Approval Status', 'Employment Years by Status'],
    [['#27ae60','#e74c3c'], ['#27ae60','#e74c3c'], ['#27ae60','#e74c3c']]
):
    for target_val, color, label in zip([1,0], color_map, ['Approved','Rejected']):
        subset = df[df['TARGET'] == target_val][col]
        ax.hist(subset, bins=20, alpha=0.65, color=color, label=label, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Preprocessing — Encoding & Scaling

In [ ]:
CATEGORICAL_COLS = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'
]
NUMERIC_COLS = [
    'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AGE_YEARS', 'YEARS_EMPLOYED',
    'IS_UNEMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
    'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'INCOME_PER_MEMBER',
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'
]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

# Fill missing OCCUPATION_TYPE
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')
print(f'OCCUPATION_TYPE unique values: {df["OCCUPATION_TYPE"].nunique()}')

# Label Encoding
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes encoded')

# Standard Scaling
scaler = StandardScaler()
df[NUMERIC_COLS] = scaler.fit_transform(df[NUMERIC_COLS])
print('\nScaling applied to numeric features.')

# Train / test split
X = df[FEATURE_COLS]
y = df['TARGET']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train approval rate: {y_train.mean()*100:.1f}%')
print(f'Test  approval rate: {y_test.mean()*100:.1f}%')

## 7. Model Training & Evaluation

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                          eval_metric='logloss', random_state=RANDOM_STATE,
                                          n_jobs=-1, verbosity=0),
}

results = {}
for name, model in MODELS.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model'   : model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc' : roc_auc_score(y_test, y_prob),
        'y_pred'  : y_pred,
        'y_prob'  : y_prob,
    }
    print(f'  Accuracy: {results[name]["accuracy"]*100:.2f}%  ROC-AUC: {results[name]["roc_auc"]:.4f}\n')

print('All models trained.')

## 8. Model Comparison

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Model'   : list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'ROC-AUC' : [r['roc_auc']  for r in results.values()],
}).set_index('Model').sort_values('Accuracy', ascending=False)

summary['Accuracy'] = summary['Accuracy'].map('{:.4f}'.format)
summary['ROC-AUC']  = summary['ROC-AUC'].map('{:.4f}'.format)
print('=== Model Performance Summary ===')
summary

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Rejected','Approved'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAccuracy: {res["accuracy"]*100:.2f}%', fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#1a3a6b', '#e74c3c', '#27ae60', '#f39c12']

ax.plot([0,1],[0,1],'k--', lw=1, label='Random Classifier')
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{name} (AUC={res['roc_auc']:.3f})")

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest and XGBoost
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, model_name in zip(axes, ['Random Forest', 'XGBoost']):
    model = results[model_name]['model']
    importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
    importances.sort_values().tail(12).plot(
        kind='barh', ax=ax,
        color='#1a3a6b' if model_name == 'Random Forest' else '#c8a84b',
        edgecolor='white'
    )
    ax.set_title(f'{model_name} — Top 12 Feature Importances', fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 9. Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Rejected','Approved']))

## 10. Best Model Selection & Save Artifacts

In [ ]:
best_name  = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']

print(f'Best Model  : {best_name}')
print(f'Accuracy    : {results[best_name]["accuracy"]*100:.2f}%')
print(f'ROC-AUC     : {results[best_name]["roc_auc"]:.4f}')

MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(best_model, os.path.join(MODEL_DIR, 'model.pkl'))
joblib.dump(scaler,     os.path.join(MODEL_DIR, 'scaler.pkl'))
joblib.dump(encoders,   os.path.join(MODEL_DIR, 'encoder.pkl'))

comparison = {name: {'accuracy': r['accuracy'], 'roc_auc': r['roc_auc']}
              for name, r in results.items()}
joblib.dump(
    {'comparison': comparison, 'best_model': best_name, 'feature_cols': FEATURE_COLS},
    os.path.join(MODEL_DIR, 'metadata.pkl')
)

print('\nArtifacts saved to ../models/')
print('  model.pkl, scaler.pkl, encoder.pkl, metadata.pkl')

## 11. Conclusions & Business Insights

### Key Findings

1. **Best Model**: The top-performing model achieved the highest accuracy and ROC-AUC among all four classifiers, making it suitable for real-world credit screening.

2. **Most Important Features**:
   - `INCOME_PER_MEMBER` — normalized income is a strong predictor of repayment capacity.
   - `AGE_YEARS` — older applicants tend to have more stable financial behavior.
   - `YEARS_EMPLOYED` — longer employment correlates with creditworthiness.
   - `NAME_INCOME_TYPE` — income source type significantly affects risk assessment.

3. **Class Distribution**: ~70% approved, ~30% rejected. Stratified splitting was used to preserve this ratio in train/test sets.

4. **Feature Engineering Impact**: Converting raw `DAYS_BIRTH` / `DAYS_EMPLOYED` to interpretable years, handling the `365243` sentinel for unemployed, and computing per-member income all improved model performance.

### Business Scenarios
- **Scenario 1 (Bank Analyst)**: Instant screening → prioritize borderline cases for manual review.
- **Scenario 2 (Compliance Officer)**: Batch-screen applicants with past payment issues.
- **Scenario 3 (Customer Self-Service)**: Let customers check eligibility before applying.

### Next Steps
- Hyperparameter tuning (GridSearchCV / Optuna) to push accuracy higher.
- Try class-weight balancing or SMOTE for better recall on rejected class.
- IBM Watson ML deployment for cloud-hosted real-time predictions.